[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/text-classification-practice/02_keras_text/02_keras_text.ipynb)

# 02. 같은 문제를 딥러닝으로 — Keras 텍스트 분류

[01번](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/text-classification-practice/01_text_baseline/01_text_baseline.ipynb)에서
TF-IDF + 로지스틱 회귀로 목표(0.70)를 훌쩍 넘겼습니다. 그런데 시험 문제지는 제출물로
**`본인핸드폰번호_2.h5`** 를 예시로 듭니다. `.h5`는 Keras 모델 파일의 확장자입니다.
"다른 확장자도 가능"하다고 적혀 있지만, **신경망으로 푸는 것을 전제한 문제**라고 보는 게 맞습니다.

## 이 장을 배우는 이유

이 노트북은 같은 데이터를 Keras로 다시 풉니다. 목표는 두 가지입니다.

1. **텍스트를 신경망에 넣는 방법**을 익힌다 — 정수 시퀀스, 패딩, [임베딩](https://github.com/karzit/temp/blob/master/glossary.md#embedding)
2. **딥러닝이 항상 이기지는 않는다**는 것을 직접 확인한다

## TF-IDF와 임베딩은 무엇이 다른가

| | TF-IDF (01번) | 임베딩 (이 노트북) |
|---|---|---|
| 입력 표현 | 사전 크기만큼의 **희소 벡터** (대부분 0) | 단어마다 **길이 64짜리 조밀 벡터** |
| 벡터 값 | 통계로 계산 (등장 횟수 × IDF) | **학습으로 결정** (역전파로 갱신) |
| 어순 | 사라짐 | `Conv1D`/`LSTM`을 쓰면 **일부 반영** |
| 단어 사이 관계 | 없음 (`우유`와 `요거트`는 남남) | 비슷한 문맥의 단어가 **가까운 벡터**가 됨 |
| 데이터가 적을 때 | 강함 | 약함 (배울 것이 많아 과적합) |

**임베딩의 값은 학습으로 정해집니다.** 이것이 핵심 차이입니다. 대신 정해야 할 값이 훨씬 많아서,
데이터가 적으면 그 자유도가 그대로 [과적합](https://github.com/karzit/temp/blob/master/glossary.md#overfitting)이 됩니다.

## 이 노트북의 구성

| 절 | 내용 |
|---|---|
| 1~2 | 텍스트 → 정수 시퀀스 → 패딩, 라벨 인코딩 |
| 3 | 가장 단순한 모델 (Embedding + 평균) |
| 4 | `Conv1D`, `LSTM`과 비교 |
| 5 | 01번의 TF-IDF 모델과 정면 비교 |
| 6 | **모델 저장의 함정** — `.h5`로 저장했더니 불러오기가 안 된다 |
| 7 | 제출 파일 만들기와 시험 체크리스트 |

> **소요 시간 50분쯤.** 학습 셀은 CPU에서 모델 하나에 5~10초입니다(4절에서 세 개를 연달아 학습합니다).
>
> **선수 지식.** Keras의 `Sequential`·`compile`·`fit`·`EarlyStopping`을 처음 본다면
> [tabular-ml-practice 04번](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/tabular-ml-practice/04_dnn_keras/04_dnn_keras.ipynb)을
> 먼저 보세요. 여기서는 **텍스트에 특수한 부분**만 설명합니다.

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules
print("Colab에서 실행 중:", IN_COLAB)

BASE_URL = "https://raw.githubusercontent.com/karzit/temp/master/notebooks/text-classification-practice/data"

if IN_COLAB:
    !pip install -q pandas scikit-learn matplotlib koreanize-matplotlib
    for _f in ["02_train.csv", "02_test_x.csv", "02_test_y.csv"]:
        !wget -q -O {_f} {BASE_URL}/{_f}
    DATA_DIR = "."
else:
    DATA_DIR = os.path.join("..", "data") if os.path.isdir(os.path.join("..", "data")) else "."

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

try:
    import koreanize_matplotlib  # noqa: F401
except ImportError:
    pass

RANDOM_STATE = 42
keras.utils.set_random_seed(RANDOM_STATE)  # 재현성을 위해 seed를 고정합니다

print("TensorFlow", tf.__version__, "· Keras", keras.__version__)

In [ ]:
from sklearn.model_selection import train_test_split

train = pd.read_csv(os.path.join(DATA_DIR, "02_train.csv"))
test_x = pd.read_csv(os.path.join(DATA_DIR, "02_test_x.csv"))
train = train.dropna(subset=["상품명"]).drop_duplicates().reset_index(drop=True)

X = train["상품명"].values
y_text = train["카테고리"].values

X_train, X_valid, y_train_text, y_valid_text = train_test_split(
    X, y_text, test_size=0.2, stratify=y_text, random_state=RANDOM_STATE
)
print("학습", len(X_train), "· 검증", len(X_valid))

---

## 1. 텍스트를 정수 시퀀스로

신경망은 TF-IDF처럼 "문서 하나 = 벡터 하나"로 받지 않습니다. **단어를 순서대로 늘어놓은
정수 배열**로 받습니다.

```
"한결식품 얼큰 컵라면 110g"  →  [12, 87, 5, 240]  →  [12, 87, 5, 240, 0, 0, 0, ...]
                                 정수로 바꾸고        길이를 맞춰 0으로 채움(패딩)
```

`TextVectorization` 레이어가 이 두 가지를 한 번에 합니다.

| 인자 | 뜻 | 정하는 법 |
|---|---|---|
| `max_tokens` | 사전에 담을 단어 수 | 자주 나오는 단어부터. 넘치면 `[UNK]` 처리 |
| `output_sequence_length` | 시퀀스 길이 | **단어 수 분포를 보고** 정함 |
| `standardize` | 소문자화·문장부호 제거 | 기본값이 이미 둘 다 함 |

**중요:** `adapt()`는 **학습 데이터에만** 부릅니다. 전체 데이터로 사전을 만들면
01번에서 본 [데이터 누출](https://github.com/karzit/temp/blob/master/glossary.md#data-leakage)입니다.

In [ ]:
단어수 = pd.Series([len(s.split()) for s in X_train])
print(단어수.describe().round(1))
print("\n95% 지점:", int(단어수.quantile(0.95)), "단어")

대부분 5~6단어이고 95%가 7단어 이하입니다. 길이를 넉넉히 **12**로 잡습니다.

**시퀀스 길이를 정하는 기준**은 간단합니다. 너무 짧으면 뒤가 잘려 정보를 잃고,
너무 길면 대부분이 0(패딩)이라 계산만 낭비합니다. **분위수를 보고 95~99% 지점**으로 잡으면 무난합니다.

In [ ]:
MAX_TOKENS = 5000
SEQ_LEN = 12

vectorize = layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_sequence_length=SEQ_LEN,
)
vectorize.adapt(X_train)   # 학습 데이터로만 사전을 만듭니다

vocab = vectorize.get_vocabulary()
print("사전 크기:", len(vocab))
print("앞 10개:", vocab[:10])
print("\n예시:", X_train[0])
print("→", vectorize([X_train[0]]).numpy()[0])

사전의 **0번은 패딩(빈 자리), 1번은 `[UNK]`(사전에 없는 단어)** 로 예약되어 있습니다.
예측할 때 처음 보는 브랜드가 나오면 전부 `[UNK]`(=1)이 됩니다. **학습 때 `[UNK]`를 한 번도 못 보면
실전에서 이 자리를 어떻게 다뤄야 할지 모른다**는 점을 기억해두세요
(`max_tokens`를 줄이면 학습 중에도 `[UNK]`가 생겨 이 문제가 완화됩니다).

출력된 정수 배열의 뒤쪽이 0으로 채워진 것도 확인하세요. 이것이 패딩입니다.

---

## 2. 라벨을 숫자로

카테고리는 문자열입니다. 신경망 출력층은 숫자만 압니다.
`LabelEncoder`로 `가나다순 → 0~9`로 바꿉니다.

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_text)   # 학습 데이터로 fit
y_valid = label_encoder.transform(y_valid_text)

N_CLASSES = len(label_encoder.classes_)
print(N_CLASSES, "개 카테고리")
for i, c in enumerate(label_encoder.classes_):
    print(f"  {i}: {c}")

**손실 함수는 `sparse_categorical_crossentropy`를 씁니다.** 라벨이 `[0, 3, 7, ...]`처럼
정수 하나로 되어 있을 때 쓰는 것이고, [원-핫](https://github.com/karzit/temp/blob/master/glossary.md#one-hot-encoding)으로
바꿔뒀다면 `categorical_crossentropy`입니다. 둘은 수학적으로 같고 **입력 형태만 다릅니다.**
원-핫으로 바꾸는 단계가 없으니 이쪽이 간단합니다.

`label_encoder.classes_`의 순서가 곧 출력층의 순서입니다. **예측값을 다시 문자열로 되돌릴 때
같은 인코더를 써야** 하므로, 7절에서 모델과 함께 저장합니다.

---

## 3. 모델 만들기 — 임베딩 + 평균

첫 모델은 최대한 단순하게 갑니다.

```
문자열 → TextVectorization → Embedding → GlobalAveragePooling1D → Dense → softmax
```

- **`Embedding(입력 사전 크기, 64)`**: 단어 하나를 길이 64짜리 벡터로 바꿉니다.
  이 벡터가 [역전파](https://github.com/karzit/temp/blob/master/glossary.md#backpropagation)로 학습됩니다
- **`GlobalAveragePooling1D`**: 단어 벡터 12개를 **평균 내어 하나로** 만듭니다.
  어순은 무시됩니다. 사실상 "학습되는 BoW"입니다. 길이가 다른 시퀀스를 **고정 길이 벡터 하나로
  줄이는 단계**가 반드시 필요한데, 평균이 그중 가장 단순한 방법입니다
  (뒤에 나오는 `GlobalMaxPooling1D`는 평균 대신 **각 자리의 최댓값**을 취합니다.
  "가장 강하게 반응한 신호만 남긴다"는 뜻이라, 핵심어 하나를 찾는 문제에 잘 맞습니다)
- **[`Dropout`](https://github.com/karzit/temp/blob/master/glossary.md#dropout)(0.3)**: 학습 중에 뉴런의 30%를 무작위로 끕니다. 과적합 대책입니다
- **출력층 `Dense(10, activation="softmax")`**: 카테고리 10개에 대한 확률

### `Sequential`이 아니라 함수형 API를 씁니다

`tabular-ml-practice` 04번에서는 층을 리스트로 쌓는 `Sequential`을 썼습니다. 여기서는 조금 다른
방식으로 씁니다.

```python
# Sequential — 층을 순서대로 나열
model = keras.Sequential([layers.Dense(64), layers.Dense(10)])

# 함수형 API — 입력을 만들고, 층을 함수처럼 호출해 이어 붙인다
inputs = keras.Input(shape=(1,), dtype=tf.string)   # 문자열 한 칸짜리 입력
x = vectorize(inputs)                                # 층(inputs) 형태로 통과시킴
outputs = layers.Dense(10, activation="softmax")(x)
model = keras.Model(inputs, outputs)                 # 입구와 출구를 지정해 모델 완성
```

**둘은 같은 모델을 만듭니다.** 함수형 쪽은 "입력이 어떤 자료형인지"(`dtype=tf.string`)를 지정할 수 있고,
`vectorize`처럼 **이미 만들어둔 레이어를 중간에 끼워 넣기** 편해서 여기서 씁니다.
`x = 층(x)`는 "x를 이 층에 통과시킨 결과를 다시 x라고 부른다"는 뜻일 뿐입니다.

`TextVectorization`을 **모델 안에 넣었다**는 점을 기억해두세요. 그러면 모델이 문자열을 그대로 받으므로
예측할 때 전처리를 다시 재현할 필요가 없습니다. (6절에서 이 선택의 대가를 봅니다.)

In [ ]:
inputs = keras.Input(shape=(1,), dtype=tf.string)   # 문자열 한 칸이 입력
x = vectorize(inputs)                                # → 정수 12개
x = layers.Embedding(MAX_TOKENS, 64, name="embedding")(x)   # → 12 × 64 벡터
x = layers.GlobalAveragePooling1D()(x)               # → 64 벡터 하나로
x = layers.Dropout(0.3)(x)
x = layers.Dense(64, activation="relu")(x)
outputs = layers.Dense(N_CLASSES, activation="softmax")(x)

model = keras.Model(inputs, outputs)
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

**파라미터 수를 보세요.** 대부분이 `Embedding` 층에 있습니다(사전 5,000 × 64차원 = 320,000개).
학습 데이터는 4,000건 남짓인데 학습할 값은 32만 개입니다. **과적합을 각오해야 하는 비율**이라
`Dropout`과 `EarlyStopping`이 선택이 아니라 필수입니다.

> 실제 사전 크기는 1,000개 미만이라(1절 출력) 임베딩 행렬의 대부분은 쓰이지 않습니다.
> `max_tokens`를 실제 사전 크기에 맞추면 모델이 훨씬 작아집니다 — 연습 문제 2번.

이제 학습합니다. `epochs=30`으로 넉넉히 잡되, **검증 손실이 3 epoch 동안 나아지지 않으면
`EarlyStopping`이 멈춥니다.** 30번을 다 돌지 않고 20 언저리에서 멈추면 정상입니다.

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    epochs=30,
    batch_size=64,
    callbacks=[early_stop],
    verbose=0,
)

hist = pd.DataFrame(history.history)   # epoch별 loss/accuracy 기록
print("학습한 epoch 수:", len(hist))
print("검증 정확도 최고: %.4f" % hist["val_accuracy"].max())

숫자만으로는 **언제 과적합이 시작됐는지** 알 수 없습니다. epoch별 기록을 그림으로 봅니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
hist[["loss", "val_loss"]].plot(ax=axes[0], title="손실")
hist[["accuracy", "val_accuracy"]].plot(ax=axes[1], title="정확도")
axes[0].set_xlabel("epoch")
axes[1].set_xlabel("epoch")
plt.tight_layout()
plt.show()

**학습 곡선을 읽으세요.** 학습 손실은 계속 내려가는데 검증 손실이 어느 시점부터 올라간다면
그 지점이 과적합의 시작이고, `EarlyStopping(restore_best_weights=True)`이 그 시점의 가중치를 되돌려줍니다.

---

## 4. 구조 세 가지 비교

`GlobalAveragePooling1D`은 어순을 버립니다. 어순을 보는 구조를 두 개 더 시도합니다.

| head | 하는 일 | 어순 |
|---|---|---|
| `average` | 단어 벡터를 평균 | 무시 |
| `conv` | **이웃한 3단어 묶음**에서 패턴을 찾음 (`Conv1D`) | 지역적으로 반영 |
| `lstm` | 앞에서 뒤로(그리고 뒤에서 앞으로) 읽으며 상태를 유지 | 전체 반영 |

[`LSTM`](https://github.com/karzit/temp/blob/master/glossary.md#lstm-gru)은 [RNN](https://github.com/karzit/temp/blob/master/glossary.md#rnn)의 개선판으로,
단어를 하나씩 읽으면서 **지금까지 읽은 내용을 상태로 들고 갑니다**(원리는
[ml-curriculum 06번](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/ml-curriculum/06_rnn/06_rnn.ipynb)에서 다룹니다).
`Bidirectional`로 감싸면 **앞→뒤와 뒤→앞을 둘 다** 읽어 두 결과를 이어 붙입니다.
"뒤에 나올 단어가 앞 단어의 뜻을 정하는" 경우까지 잡으려는 것입니다.

[CNN](https://github.com/karzit/temp/blob/master/glossary.md#cnn)을 이미지가 아니라 **텍스트에 1차원으로** 쓰는 것이 `Conv1D`입니다.
이미지에서 3×3 필터가 인접 픽셀을 보듯, 여기서는 인접한 3개 단어를 봅니다.

세 번 비슷한 코드를 쓰지 않도록, 3절에서 만든 모델을 **함수로 묶고 가운데 한 부분만 바꿉니다.**
입력부(`Input` → `TextVectorization` → `Embedding`)와 출력부(`Dropout` → `Dense` → `softmax`)는
그대로이고, **`head` 인자가 정하는 가운데만** 달라집니다.

In [ ]:
def build_model(head="average"):
    """head: average | conv | lstm — 가운데 부분만 바꿔 모델을 만든다."""
    inputs = keras.Input(shape=(1,), dtype=tf.string)
    x = vectorize(inputs)
    x = layers.Embedding(MAX_TOKENS, 64, name="embedding")(x)

    # ↓ 여기만 다르다
    if head == "average":
        x = layers.GlobalAveragePooling1D()(x)          # 평균 (3절과 동일)
    elif head == "conv":
        x = layers.Conv1D(128, 3, activation="relu")(x)  # 이웃 3단어 묶음에서 패턴 찾기
        x = layers.GlobalMaxPooling1D()(x)               # 가장 강한 신호만 남기기
    elif head == "lstm":
        x = layers.Bidirectional(layers.LSTM(64))(x)     # 앞뒤로 읽으며 상태 유지
    # ↑ 여기까지

    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation="relu")(x)
    outputs = layers.Dense(N_CLASSES, activation="softmax")(x)

    model = keras.Model(inputs, outputs)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

세 구조를 같은 조건(같은 seed, 같은 `EarlyStopping`)으로 학습해 나란히 비교합니다.
**모델 하나에 5~10초** 걸립니다.

In [ ]:
import time

results = {}
for head in ["average", "conv", "lstm"]:
    keras.utils.set_random_seed(RANDOM_STATE)
    started = time.time()

    m = build_model(head)
    h = m.fit(
        X_train, y_train,
        validation_data=(X_valid, y_valid),
        epochs=30, batch_size=64,
        callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=3,
                                                 restore_best_weights=True)],
        verbose=0,
    )
    # evaluate는 compile에 넣은 순서대로 [손실, 정확도]를 돌려줍니다. [1]이 정확도입니다.
    val_acc = m.evaluate(X_valid, y_valid, verbose=0)[1]
    results[head] = (m, val_acc)
    print(f"{head:<8} 검증 정확도 {val_acc:.4f}  ({len(h.history['loss'])} epoch, {time.time() - started:.0f}초)")

**세 구조의 차이가 거의 없습니다.** 이유는 데이터에 있습니다.

- 상품명은 **5~6단어짜리 명사 나열**이라 어순에 담긴 정보가 거의 없습니다.
  "매운 치즈 라면"과 "치즈 매운 라면"은 같은 상품입니다
- `LSTM`은 문장처럼 **긴 의존 관계**가 있는 텍스트(리뷰, 뉴스)에서 값어치를 합니다.
  여기서는 계산만 두 배로 쓰고 얻는 것이 없습니다

**입력 데이터의 성격이 구조 선택을 결정합니다.** "LSTM이 더 좋은 모델"이라는 것은 없습니다.

---

## 5. 01번의 TF-IDF 모델과 정면 비교

같은 데이터, 같은 분할로 01번의 모델을 다시 학습해 나란히 놓습니다.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline

tfidf_model = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3)),
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
).fit(X_train, y_train_text)

tfidf_acc = accuracy_score(y_valid_text, tfidf_model.predict(X_valid))

print(f"{'TF-IDF + 로지스틱 회귀':<24} {tfidf_acc:.4f}")
for head, (m, acc) in results.items():
    print(f"{'신경망 (' + head + ')':<24} {acc:.4f}")

**딥러닝이 이기지 못했습니다.** 01번의 선형 모델과 비슷하거나 조금 낮습니다.
`tabular-ml-practice` 04번에서 신경망이 랜덤 포레스트를 못 이겼던 것과 같은 장면입니다.

**왜 그럴까요?**

| 조건 | 이 문제 | 딥러닝이 유리한 상황 |
|---|---|---|
| 데이터 양 | 4천 건 | 수만~수백만 건 |
| 텍스트 길이 | 5~6단어 | 문장·문단 |
| 어순의 중요도 | 거의 없음 | 문맥이 뜻을 바꿈 ("별로 안 좋진 않다") |
| 필요한 신호 | **핵심어 하나** (`컵라면`) | 여러 단어의 조합과 뉘앙스 |

**핵심어 하나만 찾으면 되는 문제**에서는 TF-IDF가 이미 최적에 가깝습니다.
임베딩이 "단어 사이의 의미 관계"를 배울 여지는 데이터가 많아야 생깁니다.

**그럼 딥러닝은 언제 쓰나요?** 텍스트가 길고 많을 때, 그리고 **사전 학습 모델**을 쓸 때입니다.
KoBERT·KLUE-RoBERTa 같은 한국어 사전 학습 모델은 이미 대량의 한국어로 훈련되어 있어,
데이터가 적어도 강력합니다. 다만 시험 환경에서 모델을 내려받기 어렵고 학습이 느립니다.

> **시험 전략.** 먼저 TF-IDF + 선형 모델로 **목표 성능을 확보**하고, 시간이 남으면 신경망을 시도해
> 더 나은 쪽을 제출하세요. 처음부터 큰 모델을 붙잡고 있다가 시간이 부족해지는 것이 가장 나쁜 경우입니다.

---

## 6. 모델 저장의 함정

문제지는 `본인핸드폰번호_2.h5`를 예로 듭니다. 저장해보겠습니다.

In [ ]:
best_head = max(results, key=lambda k: results[k][1])
best_model = results[best_head][0]
print("가장 좋았던 구조:", best_head)

best_model.save("temp_model.h5")     # 저장은 됩니다 (경고가 나올 수 있습니다)
print("저장 완료:", round(os.path.getsize("temp_model.h5") / 1024), "KB")

try:
    reloaded = keras.models.load_model("temp_model.h5")
    print(reloaded.predict(X_valid[:3], verbose=0).argmax(axis=1))
except Exception as e:
    print("\n불러오기 실패:", type(e).__name__)
    print(str(e).split("\n")[0])

**저장은 되는데 불러오기가 실패합니다.**

원인은 `TextVectorization`입니다. 이 레이어가 들고 있는 것은 가중치가 아니라 **단어 사전(문자열 표)** 인데,
`.h5`는 숫자 배열만 담는 옛 포맷이라 그 사전을 온전히 복원하지 못합니다.

**시험에서 이것은 곧바로 감점입니다.** 채점은 "제출한 코드로 실행했을 때 제출한 모델과 동일하게
재현되는지"를 확인합니다. 불러올 수 없는 모델 파일은 그 확인을 통과하지 못합니다.

**해결책은 두 가지입니다.**

| 방법 | 하는 일 | 파일 |
|---|---|---|
| **A. `.keras`로 저장** | 최신 Keras 포맷은 전처리 레이어를 그대로 담는다 | `..._2.keras` 하나 |
| **B. 벡터화를 모델 밖으로** | 모델은 정수 배열만 받게 하고, 사전은 따로 저장 | `..._2.h5` + 사전 파일 |

문제지가 **"다른 확장자의 모델 파일"도 허용**하므로 A가 간단합니다.
B는 `.h5`를 꼭 써야 할 때의 방법입니다. 둘 다 해봅니다.

In [ ]:
# 방법 A — .keras 포맷
best_model.save("temp_model.keras")
reloaded_a = keras.models.load_model("temp_model.keras")

same = np.allclose(best_model.predict(X_valid[:20], verbose=0),
                   reloaded_a.predict(X_valid[:20], verbose=0))
print("A) .keras 불러오기 성공 · 예측 동일:", same)

이번엔 방법 B입니다. **문제의 원인은 `TextVectorization`이 모델 안에 있다는 것**이었으니,
그 레이어를 **모델 밖으로 빼면** `.h5`도 쓸 수 있습니다. 모델은 문자열 대신 **정수 배열**을 받게 되고,
그만큼 예측할 때 벡터화를 직접 해줘야 합니다. 두 단계로 나눠 진행합니다 — 먼저 학습과 모델 저장입니다.

In [ ]:
# 방법 B-1 — 벡터화를 모델 밖에서 미리 해두고, 모델은 정수 배열만 받게 만든다
X_train_ids = vectorize(X_train).numpy()
X_valid_ids = vectorize(X_valid).numpy()

keras.utils.set_random_seed(RANDOM_STATE)
int_model = keras.Sequential([
    keras.Input(shape=(SEQ_LEN,)),   # 문자열이 아니라 정수 12개를 받는다
    layers.Embedding(MAX_TOKENS, 64),
    layers.Conv1D(128, 3, activation="relu"),
    layers.GlobalMaxPooling1D(),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dense(N_CLASSES, activation="softmax"),
])
int_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
int_model.fit(X_train_ids, y_train, validation_data=(X_valid_ids, y_valid),
              epochs=30, batch_size=64, verbose=0,
              callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=3,
                                                       restore_best_weights=True)])

int_model.save("temp_int_model.h5")
print("저장 완료. 이제 사전을 따로 저장합니다.")

모델은 저장했지만 **사전은 아직 저장되지 않았습니다.** 사전은 가중치(숫자)가 아니라 문자열 목록이라
모델 파일에 담기지 않습니다. 사전이 없으면 `"한결식품 얼큰 컵라면"`을 어떤 정수로 바꿔야 할지 알 수 없어,
불러온 모델이 아무 쓸모가 없습니다. **따로 저장합니다.**

In [ ]:
# 방법 B-2 — 사전을 json으로 저장하고, 모델과 사전을 함께 되살린다
import json

# 0번 패딩, 1번 [UNK]는 TextVectorization이 자동으로 만들므로 목록에서 제외합니다.
json.dump([str(w) for w in vocab[2:]], open("temp_vocab.json", "w", encoding="utf-8"), ensure_ascii=False)

reloaded_b = keras.models.load_model("temp_int_model.h5")
restored_vectorize = layers.TextVectorization(
    max_tokens=MAX_TOKENS, output_sequence_length=SEQ_LEN,
    vocabulary=json.load(open("temp_vocab.json", encoding="utf-8")),   # 저장해둔 사전을 그대로 주입
)
print("B) .h5 불러오기 성공 · 검증 정확도 %.4f"
      % (reloaded_b.predict(restored_vectorize(X_valid), verbose=0).argmax(axis=1) == y_valid).mean())

**B의 교훈:** 모델 파일에 들어가지 않는 것(사전, 라벨 인코더, 스케일러)은 **따로 저장해야 합니다.**
표 데이터에서 "모델을 저장할 때 스케일러도 함께"였던 것과 같은 이야기입니다.
저장하는 순간이 아니라 **불러와서 예측이 되는지 확인하는 순간까지가 저장**입니다.

---

## 7. 제출 파일 만들기

시험이 요구하는 형식으로 마무리합니다. 최종 모델은 **훈련 데이터 전체**로 다시 학습합니다.

In [ ]:
import pickle

PHONE = "01012345678"  # 실제 시험에서는 본인 휴대폰 번호로

# 1) 전체 데이터로 사전과 라벨 인코더를 다시 만든다
final_vectorize = layers.TextVectorization(max_tokens=MAX_TOKENS, output_sequence_length=SEQ_LEN)
final_vectorize.adapt(X)

final_encoder = LabelEncoder()
y_all = final_encoder.fit_transform(y_text)

print("사전", len(final_vectorize.get_vocabulary()), "단어 · 카테고리", len(final_encoder.classes_), "개")

이제 4절에서 가장 좋았던 구조(`conv`)로 다시 학습합니다. **검증 세트를 떼지 않았으므로
`EarlyStopping`을 쓸 수 없습니다.** 대신 4절에서 확인한 epoch 수(9~10)를 그대로 지정합니다.
"몇 번 학습할지"를 이미 검증으로 정해뒀기 때문에 가능한 방식입니다.

In [ ]:
# 2) 같은 구조로 다시 학습 (검증 세트가 없으므로 epoch 수는 앞에서 확인한 값을 쓴다)
keras.utils.set_random_seed(RANDOM_STATE)
inputs = keras.Input(shape=(1,), dtype=tf.string)
h = final_vectorize(inputs)
h = layers.Embedding(MAX_TOKENS, 64)(h)
h = layers.Conv1D(128, 3, activation="relu")(h)
h = layers.GlobalMaxPooling1D()(h)
h = layers.Dropout(0.3)(h)
h = layers.Dense(64, activation="relu")(h)
outputs = layers.Dense(N_CLASSES, activation="softmax")(h)

final_model = keras.Model(inputs, outputs)
final_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
final_model.fit(X, y_all, epochs=10, batch_size=64, verbose=0)

# 3) 모델과 라벨 인코더 저장
#    라벨 인코더는 Keras 모델이 아니라 파이썬 객체라 모델 파일에 들어가지 않습니다.
#    pickle은 파이썬 객체를 파일로 그대로 굳혀두는 표준 모듈입니다.
final_model.save(f"{PHONE}_2.keras")
with open(f"{PHONE}_2_label_encoder.pkl", "wb") as f:
    pickle.dump(final_encoder, f)

print("저장:", f"{PHONE}_2.keras")

In [ ]:
# 4) 테스트 데이터 예측 → 제출 csv
proba = final_model.predict(test_x["상품명"].fillna("").values, verbose=0)

submission = test_x.copy()
submission["카테고리"] = final_encoder.inverse_transform(proba.argmax(axis=1))
submission.to_csv(f"{PHONE}_2.csv", index=False, encoding="utf-8-sig")

print(submission.shape)
submission.head()

In [ ]:
# 자가 채점 — 실제 시험에는 없는 단계입니다
test_y = pd.read_csv(os.path.join(DATA_DIR, "02_test_y.csv"))

nn_acc = accuracy_score(test_y["카테고리"], submission["카테고리"])
tfidf_full = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3)),
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
).fit(X, y_text)
tfidf_test_acc = accuracy_score(test_y["카테고리"], tfidf_full.predict(test_x["상품명"].fillna("")))

print("신경망       테스트 정확도 %.4f" % nn_acc)
print("TF-IDF 선형  테스트 정확도 %.4f" % tfidf_test_acc)
print("목표(0.70) 달성:", "예" if nn_acc >= 0.70 else "아니오")

### 제출 전 체크리스트

시험에서 실제로 점수를 깎이는 지점만 모았습니다.

- [ ] **파일명** — `01012345678_2.csv`처럼 숫자만. 하이픈·언더바·공백을 넣으면 0점 처리
- [ ] **모델 파일이 다시 열리는가** — 저장 후 `load_model`로 불러와 예측까지 해보기
- [ ] **모델과 함께 저장해야 할 것** — 라벨 인코더, (모델 밖에 뒀다면) 사전
- [ ] **예측 csv의 행 수**가 `02_test_x.csv`와 같은가 (결측이 있어도 `fillna("")`로 채워서 예측)
- [ ] **컬럼 구조**가 훈련 데이터와 같은가 (`상품명`, `카테고리`)
- [ ] **카테고리가 문자열**인가 (`0~9` 정수를 그대로 저장하는 실수가 흔합니다)
- [ ] **노트북을 처음부터 다시 실행해도 같은 결과**가 나오는가 — `keras.utils.set_random_seed()` 고정
- [ ] 학습 셀이 몇 분 안에 끝나는가 (채점 시 재실행합니다)

> **신경망은 완전한 재현이 어렵습니다.** seed를 고정해도 GPU 연산 순서 때문에 소수점 이하가
> 달라질 수 있습니다. 채점은 "동일하게 재현되는지"를 보되 완전 일치를 요구하지는 않지만,
> **seed 고정을 빼먹으면 예측이 눈에 띄게 달라질 수 있습니다.**

---

## 정리

- **텍스트를 신경망에 넣으려면** 정수 시퀀스 + 패딩이 필요합니다. `TextVectorization`이 둘 다 합니다
- **시퀀스 길이는 단어 수 분포의 95~99% 지점**으로 잡습니다
- **`adapt()`는 학습 데이터에만.** 사전을 전체 데이터로 만들면 데이터 누출입니다
- **임베딩은 학습되는 단어 표현**입니다. 대신 파라미터가 폭증하므로 `Dropout`·`EarlyStopping`이 필수입니다
- **구조(average/conv/lstm)의 차이는 데이터가 결정합니다.** 짧은 명사 나열에서는 어순 정보가 없어
  `LSTM`이 값어치를 못 합니다
- **딥러닝이 항상 이기지 않습니다.** 데이터 4천 건, 5단어짜리 텍스트에서는 TF-IDF + 선형 모델이 대등하거나 낫습니다
- **저장은 불러오기까지 확인해야 끝납니다.** `TextVectorization`을 품은 모델은 `.h5`로 저장하면
  다시 열리지 않습니다 — `.keras`를 쓰거나 사전을 따로 저장하세요
- **모델 파일 밖의 것**(라벨 인코더, 사전)을 같이 저장하세요

## 스스로 확인해보기

- [ ] `output_sequence_length`를 정하는 기준을 설명할 수 있다
- [ ] 사전의 0번과 1번이 무엇인지 안다
- [ ] `sparse_categorical_crossentropy`와 `categorical_crossentropy`의 차이를 안다
- [ ] `Conv1D`가 텍스트에서 무엇을 보는지 설명할 수 있다
- [ ] 이 문제에서 딥러닝이 TF-IDF를 못 이긴 이유를 세 가지 댈 수 있다
- [ ] `.h5` 저장이 실패하는 경우와 그 해결책 두 가지를 안다

## 연습 문제

풀어본 뒤 [02_keras_text_solutions.ipynb](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/text-classification-practice/02_keras_text/02_keras_text_solutions.ipynb)에서 확인하세요.

**문제 1.** `SEQ_LEN`을 4로 줄이면 성능이 어떻게 되나요? 상품명의 앞 4단어만 남는다는 뜻인데,
왜 그렇게까지 나빠지지 않는지(혹은 나빠지는지) 데이터를 보고 설명하세요.

**문제 2.** `MAX_TOKENS`를 실제 사전 크기(1절 출력)에 맞춰 줄이고 `model.summary()`의
파라미터 수를 비교하세요. 성능은 어떻게 되나요?

**문제 3.** 임베딩 차원을 16 / 64 / 256으로 바꿔가며 검증 정확도와 학습 시간을 비교하세요.
차원을 키우면 항상 좋아지나요?

**문제 4.** `TextVectorization`에 `split="character"`를 주어 **글자 단위**로 모델을 학습시켜보세요.
01번의 문자 n-gram과 같은 발상입니다. 단어 단위와 비교하면 어떤가요?
(글자 단위는 시퀀스가 길어지므로 `output_sequence_length`도 함께 늘려야 합니다.)

**문제 5.** 학습된 `Embedding` 층의 가중치를 꺼내, `라면`과 코사인 유사도가 높은 단어 10개를
찾아보세요. 의미가 비슷한 단어가 실제로 가까운지 확인하세요.

**문제 6.** 신경망의 예측 확률과 TF-IDF 모델의 예측 확률을 **평균 내어**(앙상블) 정확도를 재보세요.
두 모델이 서로 다른 오답을 낸다면 앙상블이 이득입니다. 실제로 그런가요?

---

여기까지가 이 시리즈입니다. 더 나아가고 싶다면
[시리즈 README](https://github.com/karzit/temp/blob/master/notebooks/text-classification-practice/README.md)의
"다음으로 해볼 만한 것"을 보세요 — 사전 학습 한국어 모델(KoBERT), 형태소 분석기, 계층 분류로 이어집니다.